# Context & Exposure

An agent's decisions are only as good as the information it receives. Context is the agent's global shared state — it holds everything the agent knows. Exposure strategies control *how* that state is presented to the LLM: all at once, or progressively revealed on demand.

In this tutorial, we'll build a **document analysis agent** that manages multiple skills and a growing execution history. We'll see how Exposure strategies prevent information overload while keeping details available when needed.

## Initialize

First, let's set up the LLM and define the tools our document analysis agent will use.

In [ ]:
import os
from bridgic.model import OpenAILlm

api_key = os.environ.get("OPENAI_API_KEY", "your-api-key")
base_url = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")

llm = LLM(model="openai/gpt-4o-mini")

In [ ]:
from bridgic.core import tool


@tool(description="Read the content of a document by its name")
async def read_document(doc_name: str) -> str:
    docs = {
        "quarterly_report": "Q3 Revenue: $2.5M, Growth: 15%, New customers: 120...",
        "competitor_analysis": "Main competitor launched new product, market share shift...",
        "team_updates": "Engineering: shipped v2.0, Marketing: campaign launched...",
    }
    return docs.get(doc_name, f"Document '{doc_name}' not found")


@tool(description="Summarize a text into key bullet points")
async def summarize(text: str) -> str:
    return f"Summary: Key points extracted from the given text ({len(text)} chars)"


@tool(description="Extract action items from text")
async def extract_actions(text: str) -> str:
    return f"Action items: [Review budget, Schedule follow-up, Update roadmap]"

We have three tools:

- **`read_document`** — retrieves the content of a document by name from a mock document store.
- **`summarize`** — takes a text and returns key bullet points.
- **`extract_actions`** — extracts action items from a given text.

Together, these tools let our agent read, summarize, and extract actionable insights from documents.

## Part 1: Exposure — How Data Is Disclosed to the LLM

Exposure strategies determine how context data is presented to the LLM. Bridgic Amphibious provides two strategies:

### EntireExposure

All data is visible at once. The LLM sees everything in a single prompt. This is used for `tools` in `CognitiveContext` — the agent always sees all available tool specifications.

### LayeredExposure

Progressive disclosure: the LLM first sees summaries, then can request details through the **Acquiring** cognitive policy. This is used for `skills` and `cognitive_history` — it prevents token waste on information the LLM might not need.

For `cognitive_history`, LayeredExposure creates a tiered memory system:

| Tier | What the LLM sees |
|------|--------------------|
| **Working memory** | Recent steps shown in full detail |
| **Short-term memory** | Older steps shown as summaries |
| **Long-term memory** | Oldest steps compressed into a paragraph |

In [ ]:
from bridgic.amphibious import (
    EntireExposure, LayeredExposure,
    CognitiveContext, Context,
)

# EntireExposure: everything is visible
# CognitiveContext.tools uses EntireExposure
# → LLM sees all tool specs in every prompt

# LayeredExposure: summary first, details on demand
# CognitiveContext.skills uses LayeredExposure
# → LLM sees skill names/summaries initially
# → Can request full content via Acquiring policy

# CognitiveContext.cognitive_history uses LayeredExposure
# → Recent steps shown in full (working memory)
# → Older steps shown as summaries (short-term memory)
# → Oldest steps compressed into a paragraph (long-term memory)

print("EntireExposure:", EntireExposure)
print("LayeredExposure:", LayeredExposure)

The key insight is that **EntireExposure** is simple and fast — great when the data is small (like a list of tool specs). But as data grows (like execution history), **LayeredExposure** becomes essential to avoid flooding the LLM's context window with irrelevant details.

## Part 2: Context — The Agent's Shared State

`Context` is a Pydantic `BaseModel` that automatically detects Exposure fields and provides `summary()`, `get_details()`, and `format_summary()` methods.

`CognitiveContext` is the default implementation that ships with Bridgic Amphibious. It includes the following fields:

| Field | Type | Exposure | Purpose |
|-------|------|----------|---------|
| `goal` | `str` | — | What the agent is trying to achieve |
| `tools` | `CognitiveTools` | `EntireExposure` | Available tools |
| `skills` | `CognitiveSkills` | `LayeredExposure` | Available skills |
| `cognitive_history` | `CognitiveHistory` | `LayeredExposure` | Execution history |
| `observation` | `Optional[str]` | — | Current observation |

In [ ]:
from bridgic.amphibious import CognitiveContext

# Create a context and inspect its summary
ctx = CognitiveContext(goal="Analyze quarterly documents")
print(ctx.summary())

In [ ]:
# format_summary() creates the text that the LLM actually sees
formatted = ctx.format_summary()
print(formatted)

`summary()` returns a dictionary of field summaries, while `format_summary()` produces the final text string that gets injected into the LLM prompt. This is the agent's window into its own state — anything not in this output is invisible to the LLM.

## Part 3: Custom Context — Carrying Business State

When your agent needs domain-specific state beyond goal/tools/skills/history, you can extend `CognitiveContext` with your own fields. These custom fields automatically appear in the summary shown to the LLM.

In [ ]:
from pydantic import Field, ConfigDict
from bridgic.amphibious import CognitiveContext


class DocumentAnalysisContext(CognitiveContext):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    current_document: str = Field(
        default="",
        description="Name of the document currently being analyzed"
    )
    analysis_results: dict = Field(
        default_factory=dict,
        description="Accumulated analysis results keyed by document name"
    )
    priority_level: str = Field(
        default="normal",
        description="Priority level for the current analysis task"
    )

Now let's build an agent that uses our custom context.

In [ ]:
from bridgic.amphibious import AmphibiousAutoma, CognitiveWorker, think_unit


class DocumentAnalyzer(AmphibiousAutoma[DocumentAnalysisContext]):
    analyzer = think_unit(
        CognitiveWorker.inline(
            "Read and analyze the current document. "
            "Extract key findings and action items."
        ),
        max_attempts=8,
    )

    async def on_agent(self, ctx: DocumentAnalysisContext):
        await self.analyzer

In [ ]:
agent = DocumentAnalyzer(llm=llm)
result = await agent.arun(
    goal="Analyze the quarterly report and competitor analysis, then extract action items from both",
    tools=[read_document, summarize, extract_actions],
)
print(result)

The agent uses our `DocumentAnalysisContext` to track which document it is analyzing and accumulate results. The LLM sees the custom fields in the context summary alongside the standard goal, tools, and history.

In [ ]:
# Custom fields appear in the context summary shown to the LLM
ctx = DocumentAnalysisContext(
    goal="Analyze documents",
    current_document="quarterly_report",
    priority_level="high",
)
print(ctx.format_summary())

Notice how `current_document` and `priority_level` appear in the formatted summary. The LLM now has access to this domain-specific state when making decisions, without any extra prompt engineering on your part.

<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

In this tutorial, we explored how Context and Exposure control the information flow to the LLM:

- **EntireExposure** shows all data at once — good for short lists like tool specs.
- **LayeredExposure** reveals summaries first and provides details on demand — essential for managing large skill sets and execution history without overwhelming the LLM.
- **CognitiveContext** is the default context with `goal`, `tools` (EntireExposure), `skills` (LayeredExposure), and `cognitive_history` (LayeredExposure).
- **Custom Context**: Extend `CognitiveContext` with domain-specific fields using Pydantic `Field`. These fields automatically appear in the summary shown to the LLM.
- Use `summary()` and `format_summary()` to inspect what the LLM actually sees.

Next in the advanced tutorials, we'll see how to customize the OTC cycle hooks for fine-grained control over agent behavior.